In [1]:
# clone YOLOv5 repository
!git clone https://github.com/ultralytics/yolov5.git 

fatal: destination path 'yolov5' already exists and is not an empty directory.


In [2]:
# Navigate to the cloned directory
%cd yolov5

# Install required packages
!pip install -r requirements.txt

/home/charis/Desktop/Projects/Gesture-Recognition-using-video-/yolov5


In [1]:
import torch
from IPython.display import Image, clear_output  # to display images
print('Setup complete. Using torch %s %s' % (torch.__version__, torch.cuda.get_device_properties(0) if torch.cuda.is_available() else 'GPU'))

Setup complete. Using torch 2.11.0+cu130 GPU


In [2]:
import os
os.chdir('/home/charis/Desktop/Projects/Gesture-Recognition-using-video-/sign_language.yolov5pytorch')
print(os.getcwd())  # Επαληθεύσεις ότι είσαι στη σωστή θέση

/home/charis/Desktop/Projects/Gesture-Recognition-using-video-/sign_language.yolov5pytorch


In [3]:
import yaml
with open('/home/charis/Desktop/Projects/Gesture-Recognition-using-video-/sign_language.yolov5pytorch/data.yaml', 'r') as file:
    num_classes = str(yaml.safe_load(file)['nc'])

In [4]:
num_classes

'26'

In [6]:
%cat /home/charis/Desktop/Projects/Gesture-Recognition-using-video-/yolov5/models/yolov5s.yaml

# Ultralytics 🚀 AGPL-3.0 License - https://ultralytics.com/license

# Parameters
nc: 80 # number of classes
depth_multiple: 0.33 # model depth multiple
width_multiple: 0.50 # layer channel multiple
anchors:
  - [10, 13, 16, 30, 33, 23] # P3/8
  - [30, 61, 62, 45, 59, 119] # P4/16
  - [116, 90, 156, 198, 373, 326] # P5/32

# YOLOv5 v6.0 backbone
backbone:
  # [from, number, module, args]
  [
    [-1, 1, Conv, [64, 6, 2, 2]], # 0-P1/2
    [-1, 1, Conv, [128, 3, 2]], # 1-P2/4
    [-1, 3, C3, [128]],
    [-1, 1, Conv, [256, 3, 2]], # 3-P3/8
    [-1, 6, C3, [256]],
    [-1, 1, Conv, [512, 3, 2]], # 5-P4/16
    [-1, 9, C3, [512]],
    [-1, 1, Conv, [1024, 3, 2]], # 7-P5/32
    [-1, 3, C3, [1024]],
    [-1, 1, SPPF, [1024, 5]], # 9
  ]

# YOLOv5 v6.0 head
head: [
    [-1, 1, Conv, [512, 1, 1]],
    [-1, 1, nn.Upsample, [None, 2, "nearest"]],
    [[-1, 6], 1, Concat, [1]], # cat backbone P4
    [-1, 3, C3, [512, False]], # 13

    [-1, 1, Conv, [256, 1, 1]],
    [-1, 1, nn.Upsample, [None, 2, 

In [7]:
from IPython.core.magic import register_cell_magic
@register_cell_magic
def write_yaml(line, cell):
    with open(line, 'w') as f:
        f.write(cell.format(**globals()))

In [9]:
%%writefile /home/charis/Desktop/Projects/Gesture-Recognition-using-video-/sign_language.yolov5pytorch/yolov5/models/yolov5s.yaml

# parameters
nc: {num_classes}  # number of classes
depth_multiple: 0.33  # model depth multiple
width_multiple: 0.50  # layer channel multiple

anchors:
  - [10, 13, 16, 30, 33, 23] # P3/8
  - [30, 61, 62, 45, 59, 119] # P4/16
  - [116, 90, 156, 198, 373, 326] # P5/32

# YOLOv5 v6.0 backbone
backbone:
  # [from, number, module, args]
  [
    [-1, 1, Conv, [64, 6, 2, 2]], # 0-P1/2
    [-1, 1, Conv, [128, 3, 2]], # 1-P2/4
    [-1, 3, C3, [128]],
    [-1, 1, Conv, [256, 3, 2]], # 3-P3/8
    [-1, 6, C3, [256]],
    [-1, 1, Conv, [512, 3, 2]], # 5-P4/16
    [-1, 9, C3, [512]],
    [-1, 1, Conv, [1024, 3, 2]], # 7-P5/32
    [-1, 3, C3, [1024]],
    [-1, 1, SPPF, [1024, 5]], # 9
  ]

# YOLOv5 v6.0 head
head: [
    [-1, 1, Conv, [512, 1, 1]],
    [-1, 1, nn.Upsample, [None, 2, "nearest"]],
    [[-1, 6], 1, Concat, [1]], # cat backbone P4
    [-1, 3, C3, [512, False]], # 13

    [-1, 1, Conv, [256, 1, 1]],
    [-1, 1, nn.Upsample, [None, 2, "nearest"]],
    [[-1, 4], 1, Concat, [1]], # cat backbone P3
    [-1, 3, C3, [256, False]], # 17 (P3/8-small)

    [-1, 1, Conv, [256, 3, 2]],
    [[-1, 14], 1, Concat, [1]], # cat head P4
    [-1, 3, C3, [512, False]], # 20 (P4/16-medium)

    [-1, 1, Conv, [512, 3, 2]],
    [[-1, 10], 1, Concat, [1]], # cat head P5
    [-1, 3, C3, [1024, False]], # 23 (P5/32-large)

    [[17, 20, 23], 1, Detect, [nc, anchors]], # Detect(P3, P4, P5)
  ]

Overwriting /home/charis/Desktop/Projects/Gesture-Recognition-using-video-/sign_language.yolov5pytorch/yolov5/models/yolov5s.yaml


# Train Custom YOLOv5 Detector

Here, we are able to pass a number of arguments: <br>

img: define input image size <br>
batch: determine batch size<br>
epochs: define the number of training epochs. (Note: often, 3000+ are common here!)<br>
data: set the path to our yaml file<br>
cfg: specify our model configuration<br>
weights: specify a custom path to weights. (Note: you can download weights from the Ultralytics Google Drive folder)<br>
name: result names<br>
nosave: only save the final checkpoint<br>
cache: cache images for faster training

In [10]:
%pwd

'/home/charis/Desktop/Projects/Gesture-Recognition-using-video-/sign_language.yolov5pytorch'

In [ ]:
# train yolov5s on custom data for 100 epochs
# time its performance
import time
%time

%cd /home/charis/Desktop/Projects/Gesture-Recognition-using-video-/sign_language.yolov5pytorch/

!python yolov5/train.py --img 404 --batch 160 --epochs 100 --data 'data.yaml' --cfg 'yolov5/models/yolov5s.yaml' --weights 'yolov5s.pt' --name yolov5s_results  --cache

CPU times: user 1 μs, sys: 1 μs, total: 2 μs
Wall time: 5.48 μs
/home/charis/Desktop/Projects/Gesture-Recognition-using-video-/sign_language.yolov5pytorch
train: weights=yolov5s.pt, cfg=yolov5/models/yolov5s.yaml, data=data.yaml, hyp=yolov5/data/hyps/hyp.scratch-low.yaml, epochs=100, batch_size=160, imgsz=404, rect=False, resume=False, nosave=False, noval=False, noautoanchor=False, noplots=False, evolve=None, evolve_population=yolov5/data/hyps, resume_evolve=None, bucket=, cache=ram, image_weights=False, device=, multi_scale=False, single_cls=False, optimizer=SGD, sync_bn=False, workers=8, project=yolov5/runs/train, name=yolov5s_results, exist_ok=False, quad=False, cos_lr=False, label_smoothing=0.0, patience=100, freeze=[0], save_period=-1, seed=0, local_rank=-1, entity=None, upload_dataset=False, bbox_interval=-1, artifact_alias=latest, ndjson_console=False, ndjson_file=False
github: up to date with https://github.com/ultralytics/yolov5 ✅
YOLOv5 🚀 v7.0-484-g70b964b6 Python-3.12.3 torc

In [ ]:
cd /home/charis/Desktop/Projects/Gesture-Recognition-using-video-/sign_language.yolov5pytorch/

/home/charis/Desktop/Projects/Gesture-Recognition-using-video-/sign_language.yolov5pytorch


In [ ]:
sign_language.yolov5pytorch/data.yaml